# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}\n---\n{metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll inspect the structure of the dataset. In Croissant datasets, record sets and fields are uniquely identified by `@id`. This standardized referencing makes it simple to query and extract the relevant parts of the data.

In [ ]:
# List all record sets in the dataset, referencing them by their @id
# The Dataset metadata object provides .record_sets and .fields attributes

# Show all record sets by @id and name
record_sets = dataset.metadata.record_sets
print("Record sets available:")
record_set_ids = []
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name','No name')}")
    record_set_ids.append(rs['@id'])

# Show fields per record set
print("\nFields per record set:")
for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']} ({rs.get('name','No name')})")
    for field in rs.get('fields', []):
        print(f"  Field @id: {field['@id']} Name: {field.get('name','')} DataType: {field.get('dataType','')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.
We'll extract all available record sets and display their columns using their `@id`.

In [ ]:
# Extract records from each record set using @id
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nColumns for RecordSet {record_set_id}:\n{df.columns.tolist()}")
        print(df.head())
    else:
        print(f"\nNo records found for RecordSet {record_set_id}")
if not dataframes:
    print("\nNo dataframes found. Please check the dataset structure.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**Fields referenced below use their corresponding `@id` as seen in the record set structure above.**

In [ ]:
# Let's pick the main survey record set (as example; update this selection if needed based on printed record sets)
# For demonstration, automatically pick the first record set with data
for rid, df in dataframes.items():
    if not df.empty:
        record_set_id = rid
        break
else:
    record_set_id = None
    print("No usable record set found.")

# Example numeric field: We'll try GAD-7 total score, PHQ-9 total score, or age (field @id)
numeric_field = None
for field in dataset.metadata.record_sets:
    if field['@id'] == record_set_id:
        # Search for GAD7 or PHQ9 score or age
        for f in field.get('fields', []):
            fname = f.get('name', '').lower()
            if 'gad' in fname and 'score' in fname:
                numeric_field = f['@id']
                break
            if 'phq' in fname and 'score' in fname:
                numeric_field = f['@id']
                break
            if 'age' in fname:
                numeric_field = f['@id']
                break
        if not numeric_field and field.get('fields'):
            numeric_field = field.get('fields')[0]['@id']  # fallback to first
        break

print(f"Using record set: {record_set_id}\nUsing numeric field: {numeric_field}")

df = dataframes[record_set_id]

# Filter for records with numeric_field > threshold
threshold = 10
if numeric_field in df.columns:
    # Ensure numeric
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())
    
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Group by a categorical field: try gender or education
    group_field = None
    for f in dataset.metadata.record_sets:
        if f['@id'] == record_set_id:
            for field_info in f.get('fields', []):
                name_l = field_info.get('name','').lower()
                if 'gender' in name_l:
                    group_field = field_info['@id']
                    break
                if 'education' in name_l:
                    group_field = field_info['@id']
                    break
            if not group_field and f.get('fields'):
                group_field = f.get('fields')[0]['@id']  # fallback
            break
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print(f"Field {numeric_field} not found in columns for record set {record_set_id}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below are example visualizations, such as histograms of numeric fields and grouped bar plots by categorical field, using their `@id`.

In [ ]:
# Plot histogram of the numeric field
if numeric_field in df.columns:
    plt.figure(figsize=(7,5))
    df[numeric_field].dropna().hist(bins=15, color='skyblue')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.title(f'Histogram for {numeric_field}')
    plt.show()
else:
    print(f"No data for histogram: Field {numeric_field} not in DataFrame.")

# Grouped bar plot if possible
if 'group_field' in locals() and group_field and group_field in filtered_df.columns:
    plt.figure(figsize=(7,5))
    grouped_df.set_index(group_field)[numeric_field].plot(kind='bar', color='coral')
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated how to use the `mlcroissant` library to load metadata, enumerate record sets, extract tabular records, and perform simple EDA.
- All dataset elements were referenced cleanly by their `@id` per Croissant standards.
- The dataset provides a rich set of survey data with both demographic and clinical fields for mental health research.
- Further analyses may expand on correlations, trends, or predictive modeling using these AI-ready standards.